In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaLLM
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_core.stores import InMemoryStore 
from langchain_classic.retrievers import ParentDocumentRetriever

import langchain
print(langchain.__version__)

import os

1.3.1


In [11]:
llm = OllamaLLM(model="llama3.2")

embeddings_model = OllamaEmbeddings(model="llama3.2")

In [12]:
# Carregar o pdf
pdf_link="os-sertoes.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [13]:
# Child Splitter
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200
)


parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

In [16]:
# Storages
store = InMemoryStore()

vectorstore = Chroma(
    embedding_function=embeddings_model,
    persist_directory="sertoes_child_vector_parent_rag"
)

In [20]:
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

# Processar em lotes menores
BATCH_SIZE = 100

for i in range(0, len(pages), BATCH_SIZE):
    batch = pages[i:i + BATCH_SIZE]
    print(f"Processando lote {i//BATCH_SIZE + 1} de {len(pages)//BATCH_SIZE + 1}...")
    parent_retriever.add_documents(batch, ids=None)

print("Indexação completa!")

Processando lote 1 de 7...
Processando lote 2 de 7...
Processando lote 3 de 7...
Processando lote 4 de 7...
Processando lote 5 de 7...
Processando lote 6 de 7...
Processando lote 7 de 7...
Indexação completa!


In [24]:
def ask(question):
    prompt = ChatPromptTemplate.from_template("""
    Responda a pergunta com base apenas no contexto abaixo:
    
    <context>
    {context}
    </context>
    
    Pergunta: {question}
    """)

    docs = parent_retriever.invoke(question)
    for i, doc in enumerate(docs):
        print(f"--- Chunk {i+1} ---")
        print(doc.page_content)
        print(f"Metadata: {doc.metadata}")
        print()
    
    # Chain no padrão LCEL (versão 1.x)
    chain = (
        {"context": parent_retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain.invoke(question)

In [25]:
user_question = input("User: ")
answer = ask(user_question)
print("Answer: ", answer)

User:  Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?


--- Chunk 1 ---
biblioteca básica brasileira – cultive um livro
xxviii
artísticas que iam do ideário da Revolução Francesa e sua ado -
ção pelo Romantismo, até aquelas contaminadas pela ciência do 
tempo, como o positivismo, o realismo e o naturalismo. Assim, 
Os	 sertões  representou o mea	 culpa  da geração de Euclides que, 
como membro da consciência letrada do país, não compreendera 
aquele Brasil profundo, esquecido pelos governantes do litoral. 
Contudo, não custa ressaltar que o livro de Euclides da Cunha 
não era a defesa da vida sertaneja contra a citadina, pois, para 
ele, “Estamos condenados à civilização. Ou progredimos, ou 
desaparecemos.” 
Euclides da Cunha nasceu em uma fazenda no município 
fluminense de Cantagalo, em 1866, e morreu na cidade do Rio de 
Janeiro, em 1909. O homem múltiplo – militar, jornalista, enge -
nheiro, escritor, militante político, explorador da Amazônia – 
teve uma infância de menino órfão, criado por tias, perambu -
lando por fazendas. O adolesc